# 7주차 — LangSmith (2) 데이터셋과 평가 (Colab판)

「최신인공지능」 2026 · 7주차 실습 · 2026년 10월 16일 (금)

> **"논문에 따르면 CoT가 낫다 — 그런데 *내 과제에서도* 그렇습니까? 오늘 재봅니다."**

| 실습 | 교시 | 내용 |
|------|------|------|
| 사전 🔶 | — | `evaluate()` / `create_examples()` 시그니처 확인 ★★ |
| 배포본 | 1교시 | 데이터셋 16건 (대표 7 + **엣지 9**) · 라벨링 기준 ★ |
| 실습 1 ★ | 1교시 | LangSmith 데이터셋 생성 |
| — | 2교시 | 규칙 기반 평가자 3종 + **`contains` 의 함정** ★★ |
| — | 2교시 | LLM 판정자 — **판정자를 검증한다** ★★ |
| 실습 2 ★★ | 2교시 | **CoT vs Self-Consistency — 정확도·토큰·지연** |
| 실습 3 | 3교시 | 프롬프트 수정 → 재평가 → **회귀 찾기** |
| — | 3교시 | 무료 한도 계산 |

> ### ⏱ 실행 시간 주의 — 이 노트북은 오래 걸립니다
>
> 실습 2는 예제 16개 × (1회 + 5회) = **96회 호출**입니다.
> **[런타임] > [런타임 유형 변경] > T4 GPU** 를 반드시 먼저 설정하십시오.
> CPU 런타임에서는 실습 2가 끝나지 않습니다.
>
> ★ **실행을 먼저 걸어 두고**, 도는 동안 아래 해석 틀을 읽으십시오.

## 0. 환경 준비

6주차와 같은 LangSmith 키를 씁니다. **새 키는 없습니다.**
`LANGSMITH_PROJECT` 만 `week07-eval` 로 바꿉니다.

> ⚠️⚠️ **개인 신용카드를 절대 등록하지 마십시오.**
> 무료 Developer 플랜은 카드 없이 가입되고, 한도를 넘으면
> **429로 거부될 뿐 과금되지 않습니다**(하드 캡).

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Colab 환경 준비 — 매 세션 1회 실행 (재실행 안전)
# ══════════════════════════════════════════════════════════════
WEEK_MODELS   = ["chat"]
WEEK_PACKAGES = ("langchain langchain-core langchain-ollama langchain-openai "
                 "python-dotenv pydantic langsmith")
WEEK_SECRETS  = ["LANGSMITH_API_KEY", "OPENAI_API_KEY"]   # OpenAI 는 없어도 진행됩니다

# ──────────────────────────────────────────────────────────────
import os, shutil, subprocess, sys, time, urllib.request

IN_COLAB = "google.colab" in sys.modules
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

GPU   = shutil.which("nvidia-smi") is not None and sh("nvidia-smi").returncode == 0
CHAT  = os.environ.setdefault("MODEL",       "gemma3:4b" if GPU else "gemma3:1b")
SMALL = os.environ.setdefault("SMALL_MODEL", "gemma3:1b")
EMBED = os.environ.setdefault("EMBED_MODEL", "nomic-embed-text")
TOOL  = os.environ.setdefault("TOOL_MODEL",  "qwen3:4b")
PICK  = {"chat": CHAT, "small": SMALL, "embed": EMBED, "tool": TOOL}

print(f"[1/5] 런타임   {'GPU 있음 ✅' if GPU else 'CPU 전용 ⚠️'}   →  대화 모델 {CHAT}")
if not GPU:
    print("       ⚠️⚠️ 이 차시는 96회 호출이 필요합니다. CPU 런타임에서는 끝나지 않습니다.")
    print("       [런타임] > [런타임 유형 변경] > T4 GPU 로 바꾸고 다시 실행하십시오.")

print("[2/5] 패키지 설치 중…")
r = sh(f"{sys.executable} -m pip install -q {WEEK_PACKAGES}")
print("       ✅ 완료" if r.returncode == 0 else "       ❌ 실패\n" + r.stderr[-600:])

if shutil.which("ollama") is None:
    print("[3/5] Ollama 설치 중… (약 30초)")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
print("[3/5] Ollama  " + ("✅ 준비됨" if shutil.which("ollama") else "❌ 설치 실패"))

def alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not alive():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if alive():
            break
        time.sleep(1)
print("[4/5] 서버    " + ("✅ 응답함" if alive() else "❌ 미응답 — 이 셀을 다시 실행하세요"))

have = {ln.split()[0] for ln in sh("ollama list").stdout.splitlines()[1:] if ln.strip()}
for key in WEEK_MODELS:
    name = PICK[key]
    if name in have:
        print(f"[5/5] {name:<20s} ✅ 이미 있음")
        continue
    print(f"[5/5] {name:<20s} ⏳ 내려받는 중… (진행 표시 없이 수 분 걸립니다)")
    t0 = time.time()
    r = sh(f"ollama pull {name}")
    print(f"       {'✅ 완료' if r.returncode == 0 else '❌ 실패'}  ({time.time() - t0:.0f}초)")

# ── LangSmith 설정 (6주차와 같은 키) ─────────────────────────
for k in WEEK_SECRETS + ["LANGSMITH_PROJECT", "LANGSMITH_TRACING"]:
    if not os.getenv(k) and IN_COLAB:
        try:
            from google.colab import userdata
            os.environ[k] = userdata.get(k)
        except Exception:
            pass

os.environ["LANGSMITH_PROJECT"] = "week07-eval"      # ★ 6주차와 프로젝트를 나눈다
os.environ.setdefault("LANGSMITH_TRACING", "true")

print()
print("[키]  LANGSMITH_API_KEY   " + ("✅ 설정됨" if os.getenv("LANGSMITH_API_KEY") else "❌ 없음 — 🔑 보안 비밀에 등록하세요"))
print("[키]  OPENAI_API_KEY      " + ("✅ 설정됨" if os.getenv("OPENAI_API_KEY") else "⬜ 없음 (로컬 판정자만 씁니다)"))
print(f"[설정] LANGSMITH_PROJECT  = {os.getenv('LANGSMITH_PROJECT')}")

print("\n" + "=" * 62)
print(f"준비 완료 — MODEL='{CHAT}'")
print("=" * 62)

## 사전 점검 🔶 — 이 차시 최대의 위험 지점

`langsmith` 는 버전 변화가 빠르고, 아래 두 가지가 버전마다 다릅니다.

```
① create_examples() 의 인자 형태
     inputs=/outputs= 리스트   vs   examples=[{...}] dict 리스트
② 평가자 함수의 시그니처
     (outputs, reference_outputs)   vs   (run, example)
```

**여기서 막히면 2교시 25분 실습이 통째로 멈춥니다.**

아래 셀은 **LLM 없이** 더미 target 으로 `evaluate()` 를 1회 실제 실행해 보고,
어느 형태가 동작하는지 확정한 뒤 임시 데이터셋을 지웁니다. (traces 2건 소모)

In [ ]:
import inspect, os

TEMP_DATASET = "week07-apicheck-tmp"


def section(title: str) -> None:
    print("\n" + "─" * 70)
    print(f"  {title}")
    print("─" * 70)


# ── ① 설치된 버전 ──
section("① 설치된 버전")
from importlib.metadata import version
for pkg in ("langsmith", "langchain", "langchain-core", "langchain-ollama", "langchain-openai"):
    try:
        print(f"  {pkg:<20} {version(pkg)}")
    except Exception:
        print(f"  {pkg:<20} (미설치)")

# ── ② evaluate 의 import 경로 ──
section("② evaluate 의 import 경로")
try:
    from langsmith import evaluate
    print("  [O] from langsmith import evaluate             ← 신형")
except ImportError:
    from langsmith.evaluation import evaluate
    print("  [O] from langsmith.evaluation import evaluate  ← 구형 🔶")

from langsmith import Client
client = Client()

# ── ③ create_examples() 가 받는 인자 ──
section("③ create_examples() 가 받는 인자")
params = list(inspect.signature(client.create_examples).parameters)
print(f"  {params}")
if "examples" in params:
    print("  [O] 현재 형태 (examples=[{...}] dict 리스트) 사용 가능 ★")
if "inputs" in params and "outputs" in params:
    print("  [O] 레거시 형태가 명시적 인자로 남아 있습니다 — 구버전")
elif "kwargs" in params:
    print("  [i] 레거시 형태는 **kwargs 로만 받습니다 (0.3.11 이후) 🔶")

# ── ④ evaluate() 실제 실행 — LLM 없이 ──
section("④ evaluate() 실제 실행 — LLM 없이 더미 target 으로")
for existing in client.list_datasets(dataset_name=TEMP_DATASET):
    client.delete_dataset(dataset_id=existing.id)

dataset = client.create_dataset(dataset_name=TEMP_DATASET, description="API 확인용 임시")
pairs = [({"review": "좋아요"}, {"label": "긍정"}), ({"review": "별로"}, {"label": "부정"})]
try:
    client.create_examples(dataset_id=dataset.id,
                           examples=[{"inputs": i, "outputs": o} for i, o in pairs])
    print("  [O] create_examples 현재 형태(examples=)로 등록 성공 ★")
except (TypeError, ValueError):
    client.create_examples(inputs=[i for i, _ in pairs],
                           outputs=[o for _, o in pairs], dataset_id=dataset.id)
    print("  [O] create_examples 레거시 형태로 등록 성공 🔶")


def dummy_target(inputs: dict) -> dict:
    """LLM 을 부르지 않는다. 첫 글자로 대충 답한다 — API 형태만 확인하면 된다."""
    return {"label": "긍정" if "좋" in inputs["review"] else "부정"}


def new_style(outputs: dict, reference_outputs: dict) -> bool:
    return outputs["label"] == reference_outputs["label"]


def old_style(run, example):
    return {"key": "exact_match",
            "score": (run.outputs or {}).get("label") == (example.outputs or {}).get("label")}


EVAL_STYLE = None
try:
    list(evaluate(dummy_target, data=TEMP_DATASET, evaluators=[new_style],
                  experiment_prefix="apicheck-new", max_concurrency=1))
    EVAL_STYLE = "신형 (outputs, reference_outputs)"
    print("  [O] 신형 평가자 시그니처 동작 ★  — 아래 코드를 그대로 쓰면 됩니다")
except Exception as e:
    print(f"  [X] 신형 실패: {type(e).__name__}: {e}")
    try:
        list(evaluate(dummy_target, data=TEMP_DATASET, evaluators=[old_style],
                      experiment_prefix="apicheck-old", max_concurrency=1))
        EVAL_STYLE = "구형 (run, example)"
        print("  [O] 구형 평가자 시그니처 동작 🔶 — as_legacy() 로 감싸서 넘깁니다")
    except Exception as e2:
        print(f"  [X] 구형도 실패: {type(e2).__name__}: {e2}")

for existing in client.list_datasets(dataset_name=TEMP_DATASET):
    client.delete_dataset(dataset_id=existing.id)
print(f"\n  [i] 임시 데이터셋 '{TEMP_DATASET}' 삭제 완료")

section("결론")
print(f"  ✅ 동작하는 평가자 시그니처 : {EVAL_STYLE or '⚠️ 어느 쪽도 동작하지 않음'}")

## 1교시 — 데이터셋 배포본

```
Example = 입력(Input) + 기대 출력(Reference Output)

   Example 1   입력: "배송이 하루 만에 왔어요"        기대: "긍정"
   Example 8   입력: "나쁘지 않네요"                  기대: "긍정"   ← 엣지 ★
```

> ### ★★ 좋은 데이터셋의 조건
>
> **쉬운 것만 넣으면 모든 실험이 100점이 나와 비교가 안 됩니다.**
>
> - **대표 케이스**: 전형적인 입력. 이게 틀리면 큰일 (절반)
> - **엣지 케이스**: 애매·이중부정·반어·양가·빈 입력 등 (절반) ★
>
> ⚠️ **라벨링 기준을 먼저 못 박아야 합니다.**
> 기준 없이 라벨을 달면 데이터셋 자체가 흔들리고, 그 위의 모든 측정이 무의미해집니다.

In [ ]:
import unicodedata

LABELS = ("긍정", "부정", "중립")


# ── 표를 그리기 위한 잡일 ──────────────────────────────────────
#    한글·★ 는 폭이 2인데 len() 은 1로 셉니다. 그대로 f"{s:<20}" 하면 표가 어긋납니다.
def width(s: str) -> int:
    return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in str(s))


def pad(s: str, n: int, align: str = "<") -> str:
    """표시 폭 기준으로 채운다. align: '<' 왼쪽 / '>' 오른쪽."""
    s = str(s)
    fill = " " * max(0, n - width(s))
    return s + fill if align == "<" else fill + s


LABELING_RULE = """\
[라벨링 기준] — 데이터셋에 반드시 함께 남긴다 ★

  1. 긍정 / 부정 / 중립 세 가지만 쓴다.
  2. 양가(장점+단점이 함께 있는) 표현은 **마지막 절의 인상**을 따른다.
       "가격은 좋은데 품질은 실망입니다"   → 부정
       "배송은 느렸지만 제품은 완벽합니다" → 긍정
  3. 사실만 진술하고 평가가 없으면 중립.
  4. 반어·비꼼은 **말한 뜻**이 아니라 **의도한 뜻**을 따른다.
  5. 빈 입력은 중립으로 둔다 (판단 근거가 없으므로).

  ⚠️ 2번과 4번은 사람마다 갈립니다. 그래서 '적어 둡니다'.
     적어 두지 않은 기준은 다음 주에 다른 기준이 됩니다.
"""

# (입력, 기대 출력, 종류, 메모)   종류: "대표" | "엣지"
EXAMPLES = [
    # ── 대표 케이스 ─────────────────────────────────────────────
    ({"review": "배송이 하루 만에 왔어요"}, {"label": "긍정"}, "대표", "단순 칭찬"),
    ({"review": "화면에 흠집이 있네요"}, {"label": "부정"}, "대표", "단순 불만"),
    ({"review": "음질이 기대 이상입니다. 재구매 의사 있어요"}, {"label": "긍정"}, "대표", "명시적 만족"),
    ({"review": "일주일 만에 고장 났습니다. 환불 요청했어요"}, {"label": "부정"}, "대표", "명시적 불만"),
    ({"review": "설명서대로 조립했고 잘 작동합니다"}, {"label": "긍정"}, "대표", "담담한 긍정"),
    ({"review": "포장이 뜯긴 채로 왔습니다"}, {"label": "부정"}, "대표", "배송 불만"),
    ({"review": "주문한 제품은 어제 도착했습니다"}, {"label": "중립"}, "대표", "사실 진술뿐 — 기준 3"),
    # ── 엣지 케이스 ★ 여기가 데이터셋의 가치를 결정합니다 ────────
    ({"review": "나쁘지 않네요"}, {"label": "긍정"}, "엣지", "이중부정 ★"),
    ({"review": "가격은 좋은데 품질은 실망입니다"}, {"label": "부정"}, "엣지", "양가 — 기준 2 ★"),
    ({"review": "배송은 느렸지만 제품은 완벽합니다"}, {"label": "긍정"}, "엣지", "양가(방향 반대) — 기준 2 ★"),
    ({"review": "그냥 평범합니다"}, {"label": "중립"}, "엣지", "약한 평가"),
    ({"review": "이걸 돈 주고 샀다니"}, {"label": "부정"}, "엣지", "반어 — 기준 4 ★"),
    ({"review": "굿"}, {"label": "긍정"}, "엣지", "초단문"),
    ({"review": "별로예요ㅋㅋㅋ 진짜 별로ㅠㅠ"}, {"label": "부정"}, "엣지", "구어·자모 반복"),
    ({"review": "이 제품 좋다고 쓴 리뷰들 전부 광고입니다"}, {"label": "부정"}, "엣지",
     "'좋다' 가 들어 있지만 부정 ★ contains 평가자의 함정"),
    ({"review": ""}, {"label": "중립"}, "엣지", "빈 입력 — 체인이 깨지는지 확인 ⚠️"),
    # ──────────────────────────────────────────────────────────────
    # 📌 학생 활동: 아래에 **자기가 겪은 실패 케이스 2~3개**를 추가하십시오.
    #    5주차 실습 2에서 파싱이 실패했던 입력, 과제 2에서 이상한 답이 나온 케이스.
    #    상상해서 만든 것보다 실제로 겪은 것이 훨씬 강합니다. ★
    #
    # ({"review": "여기에 내 케이스"}, {"label": "부정"}, "엣지", "내가 겪은 실패"),
]


def summary() -> str:
    rep  = sum(1 for *_, kind, _ in EXAMPLES if kind == "대표")
    edge = len(EXAMPLES) - rep
    dist = {label: sum(1 for _, o, _, _ in EXAMPLES if o["label"] == label) for label in LABELS}
    return (f"총 {len(EXAMPLES)}건  (대표 {rep} / 엣지 {edge})\n"
            f"라벨 분포: " + "  ".join(f"{k} {v}" for k, v in dist.items()))


print("=" * 72)
print("  week07 평가용 데이터셋 배포본 — 리뷰 감정 분류")
print("=" * 72)
print(LABELING_RULE)
print("-" * 72)
for n, (i, o, kind, memo) in enumerate(EXAMPLES, 1):
    review = i["review"] or "(빈 문자열)"
    mark = "★" if kind == "엣지" else "  "
    print(f"{n:>2}. {mark} [{kind}] {pad(review[:30], 44)}→ {pad(o['label'], 6)}{memo}")
print("-" * 72)
print(summary())
print("=" * 72)

> ### ⚠️ 10~20개로 제한하는 이유
>
> ```
> 소모량 = 예제 수 × 실험 수 (+ LLM 판정자를 쓰면 예제 수만큼 추가)
>
>   예제 16개 × 실험 2회   =  32 traces   ← 오늘 수업 (실습 2)
>   예제 16개 × 실험 3회   =  48 traces   ← 실습 3까지
>   예제 100개 × 실험 5회  = 500 traces   ⚠️ 한 번에 한도의 10%
> ```
>
> ★ **20개짜리 데이터셋이 '하나도 없는 것' 보다 압도적으로 낫습니다.**
> 실무에서도 1,000개로 시작하지 않습니다. 20개로 시작해 실패할 때마다 늘립니다.

## 실습 1 ★ (1교시) — LangSmith 데이터셋 생성

```
6주차: 계기판을 달았다   — 무슨 일이 있었나 (관측)
7주차: 눈금을 만든다     — 얼마나 좋은가   (측정) ★

    ① 데이터셋  : 고정된 문제집 (입력 + 기대 출력)   ← 지금
    ② 평가자    : 채점 기준
    ③ 실험      : 기법을 바꿔가며 같은 문제집을 푼다
    ④ 비교      : 어느 쪽이 나은가 — 숫자로
```

데이터셋 만드는 방법은 두 가지입니다.

- ① **수동 작성** — 위 배포본 + 학생 추가분
- ② **추적 로그 수집** ★ — 웹에서 6주차 Run 을 골라 `Add to Dataset`
  (코드가 아니라 **화면**에서 합니다)

In [ ]:
DATASET_NAME = "week07-review-sentiment"
DESCRIPTION  = "리뷰 감정 분류 — 대표 + 엣지 케이스. 7주차 기법 A/B 실측용"

RESET = False       # ⚠️ True 로 두면 기존 데이터셋을 지우고 새로 만듭니다


def get_or_create(client, reset: bool):
    """데이터셋을 가져오거나 만든다. 재실행을 견디게 하는 것이 목적. ★"""
    found = list(client.list_datasets(dataset_name=DATASET_NAME))
    if found and reset:
        print(f"⚠️  RESET : 기존 '{DATASET_NAME}' 을 삭제합니다.")
        client.delete_dataset(dataset_id=found[0].id)
        found = []
    if found:
        print(f"[i] 이미 있는 데이터셋을 재사용합니다 — {DATASET_NAME}")
        return found[0], False
    ds = client.create_dataset(dataset_name=DATASET_NAME, description=DESCRIPTION)
    print(f"[+] 데이터셋 생성 — {DATASET_NAME}")
    return ds, True


def add_examples(client, dataset_id, pairs) -> int:
    """예제를 추가한다. 🔶 두 가지 인자 형태를 모두 시도한다.

    · 현재 형태 : examples=[{"inputs": ..., "outputs": ...}]   ← 먼저 시도
    · 레거시    : inputs=[...], outputs=[...]                  ← 강의안에 적힌 형태
      버전이 올라가면 먼저 끊기는 쪽은 레거시이므로 현재 형태를 기본으로 둡니다.
    """
    if not pairs:
        return 0
    try:
        client.create_examples(
            dataset_id=dataset_id,
            examples=[{"inputs": i, "outputs": o} for i, o in pairs],
        )
        return len(pairs)
    except (TypeError, ValueError) as e:
        print(f"[!] 현재 형태 실패 ({type(e).__name__}) → 레거시 형태로 재시도 🔶")
    client.create_examples(
        inputs=[i for i, _ in pairs], outputs=[o for _, o in pairs], dataset_id=dataset_id
    )
    return len(pairs)


print("=" * 72)
print("  실습 1 — 평가용 데이터셋 구성")
print("=" * 72)
print(summary())
print()

dataset, created = get_or_create(client, RESET)

# ── 이미 들어 있는 입력은 건너뛴다 (중복 등록 방지) ★ ──────────
existing = {(e.inputs or {}).get("review") for e in client.list_examples(dataset_id=dataset.id)}
if existing and not created:
    print(f"[i] 기존 예제 {len(existing)}건 — 없는 것만 추가합니다.")

todo  = [(i, o) for i, o, _, _ in EXAMPLES if i["review"] not in existing]
added = add_examples(client, dataset.id, todo)
total = len(list(client.list_examples(dataset_id=dataset.id)))

print()
print(f"[✓] {added}건 등록  →  '{DATASET_NAME}' 총 {total}건")

### 웹에서 확인합니다

`smith.langchain.com` → **Datasets & Testing** → `week07-review-sentiment`

- 예제를 **화면에서 직접 추가·수정**할 수 있습니다 (비개발자 협업 시 유용)
- `Input` / `Reference Output` 두 칸이 곧 **'문제와 정답'** 입니다

### ★★ 방법 ② — 추적 로그에서 수집 (여기서부터는 화면 작업입니다)

```
Projects → week06-tracing → Runs 목록
    ① 이상한 답이 나온 Run 을 찾는다
    ② "Add to Dataset" 으로 이 데이터셋에 추가            ★
    ③ 기대 출력(정답)을 손으로 채운다
```

| | 특징 |
|---|---|
| 수동 작성 | 내가 상상한 케이스 — 빠르지만 **내가 아는 것에 편향** |
| **로그 수집** ★ | 실제로 일어난 케이스 — **내가 예상 못 한 입력**이 들어 있다 |

> 📌 **실무에서는 ② 가 본체입니다.** 실패한 입력을 계속 담아
> 다시는 그 실패가 재발하지 않게 합니다 — 이것이 **회귀 테스트**입니다 (3교시).
>
> 📌 **학생 활동**: 위 `EXAMPLES` 리스트 아래쪽에 **내 엣지 케이스 2~3개**를 추가하고
> 두 셀을 다시 실행하십시오. (이미 등록된 것은 건너뜁니다)

## 2교시 1절 — 규칙 기반 평가자: 코드로 채점 기준을 고정한다

기대 정답이 `"긍정"` 인데 모델이 이렇게 답했습니다. **맞습니까?**

| | 답안 | 판정 |
|---|---|---|
| ① | `"긍정"` | ✅ 명백히 맞음 |
| ② | `"긍정입니다"` | ??? |
| ③ | `"  긍정  "` | ??? (공백) |
| ④ | `"이 리뷰는 긍정적입니다"` | ??? |
| ⑤ | `"약간 긍정"` | ??? |
| ⑥ | `"positive"` | ??? |

⚠️ **채점 기준을 먼저 정하지 않으면 매번 다른 점수가 나옵니다.**

| 평가자 | 잡아내는 것 | 놓치는 것 |
|---|---|---|
| 정확 일치 | 형식까지 완벽한가 | `"긍정입니다"` 를 오답 처리 ⚠️ |
| 포함 | 군더더기 허용 | **`"긍정이 아닙니다"` 를 정답 처리** ⚠️★ |
| 정규식 | 형식 준수 여부 | 내용의 옳고 그름은 못 봄 |

> ★ **이 절의 결론 한 문장: 평가자도 검증 대상입니다.**
> 평가자 자체에 버그가 있으면 그 위에서 잰 측정 결과 전체가 무의미해집니다.

In [ ]:
import re

VALID = r"(긍정|부정|중립)"


# ── ① 정확 일치 — 가장 엄격 ───────────────────────────────────
def exact_match(outputs: dict, reference_outputs: dict) -> bool:
    """모델 출력이 정답과 정확히 같은가."""
    return (str(outputs.get("label", "")).strip()
            == str(reference_outputs.get("label", "")).strip())


# ── ② 포함 — 앞뒤 군더더기를 허용 ★ ───────────────────────────
def contains(outputs: dict, reference_outputs: dict) -> bool:
    """정답 문자열이 출력 안에 들어 있는가.  ⚠️ 함정이 있다 — 아래 시연 참고."""
    return str(reference_outputs.get("label", "")).strip() in str(outputs.get("label", ""))


# ── ③ 정규식 — 형식을 검사 ────────────────────────────────────
def is_valid_label(outputs: dict, reference_outputs: dict) -> bool:
    """세 라벨 중 하나만 나왔는가 (형식 준수 여부. 정답 여부와 무관) ★"""
    return bool(re.fullmatch(VALID, str(outputs.get("label", "")).strip()))


# ── 참고: 실무에서 흔히 쓰는 절충안 ───────────────────────────
def normalized_match(outputs: dict, reference_outputs: dict) -> bool:
    """군더더기는 허용하되 부정 표현은 걸러낸다 — contains 의 함정을 막은 형태."""
    got  = str(outputs.get("label", "")).strip()
    want = str(reference_outputs.get("label", "")).strip()
    found = re.findall(VALID, got)
    if len(found) != 1 or found[0] != want:
        return False
    # ⚠️ 한국어 활용형 주의 — "아니" 는 "아닙니다" 의 부분 문자열이 아닙니다.
    #    (아/닙/니/다) 이므로 "아닙" 을 따로 넣어야 잡힙니다.
    #    규칙을 정교하게 짜기 어렵다는 것이 바로 이런 것입니다. ★
    return not re.search(r"(아니|아닙|아님|아냐|않|없|말고|반대)", got)


ALL_EVALUATORS = [exact_match, contains, is_valid_label, normalized_match]

# ── 시연 — 평가자를 평가한다 ★★ ──────────────────────────────
#   (모델 출력, 기대 정답, 사람이 보기에 맞는가)
CASES = [
    ("긍정", "긍정", True),
    ("긍정입니다", "긍정", True),
    ("  긍정  ", "긍정", True),
    ("이 리뷰는 긍정적입니다", "긍정", True),
    ("positive", "긍정", True),
    ("긍정이 아닙니다", "긍정", False),          # ★★ contains 의 함정
    ("부정", "긍정", False),
    ("애매하지만 굳이 고르면 긍정", "긍정", True),
    ("", "중립", False),                          # 빈 출력 (호출 실패)
]

print("=" * 78)
print("  규칙 기반 평가자 3종 — 같은 답안을 서로 다르게 채점한다")
print("=" * 78)
header = ["모델 출력", "정답", "사람", "정확일치", "포함", "정규식", "정규화"]
widths = [32, 6, 6, 10, 8, 8, 8]
print("".join(pad(h, w) for h, w in zip(header, widths)))
print("-" * 78)

wrong = {fn.__name__: 0 for fn in ALL_EVALUATORS}
for got, want, human in CASES:
    o, r = {"label": got}, {"label": want}
    marks = []
    for fn in ALL_EVALUATORS:
        v = fn(o, r)
        if v != human:
            wrong[fn.__name__] += 1
        marks.append("O" if v else "X")
    cells = [f'"{got}"', want, "O" if human else "X", *marks]
    print("".join(pad(c, w) for c, w in zip(cells, widths)))

print("-" * 78)
for fn in ALL_EVALUATORS:
    print(f"  {fn.__name__:<18} 사람 판정과 불일치 {wrong[fn.__name__]}/{len(CASES)}건")
print("=" * 78)

### 읽어낼 것 ★★

**①** `"긍정입니다"` 를 `exact_match` 는 **오답**으로 셉니다.
모델은 맞혔는데 평가자가 틀렸다고 셉니다 → **정확도가 실제보다 낮게** 나옵니다.

**②** `"긍정이 아닙니다"` 를 `contains` 는 **정답**으로 셉니다. ⚠️★
`"긍정이 아닙니다"` 안에 `"긍정"` 이 들어 있기 때문입니다.
모델은 틀렸는데 평가자가 맞았다고 셉니다 → **정확도가 실제보다 높게** 나옵니다.

**③** `is_valid_label` 은 **형식만** 봅니다. `"부정"` 도 형식은 통과합니다.
형식 검사와 정답 검사는 다른 축입니다
(5주차: *스키마는 형식을 보장하지만 내용은 보장하지 않는다* — 그 구분이 여기서도 그대로) ★

> → **평가자 자체에 버그가 있으면 측정 결과 전체가 무의미합니다.**
> 평가자도 이렇게 몇 건을 손으로 채점해 대조해야 합니다.

| ✅ 규칙 기반의 장점 | ⚠️ 한계 |
|---|---|
| 비용 0원 (LLM 호출 없음) | 표현이 조금만 달라도 오답 처리 |
| 즉시 채점 | **의미가 같은지는 판단 못 함** ★ |
| 항상 같은 결과 (재현 가능) | 요약·번역·생성형 과제에 적용 불가 |
| 기준이 코드로 남아 검토 가능 | 규칙을 정교하게 짜기 어렵다 |

> 📌 **규칙 기반으로 되는 일은 규칙 기반으로 하십시오.**
> 분류·추출처럼 답이 정해진 과제에 LLM 판정자를 쓸 이유가 없습니다. 돈과 시간만 더 듭니다.

In [ ]:
# ── 🔶 evaluate() 호출 헬퍼 — 평가자 시그니처가 안 맞으면 구형으로 재시도 ──
#    ⚠️ 이 차시 최대의 위험 지점입니다. 위 사전 점검 셀이 이미 확정해 두었습니다.

def as_legacy(fn):
    """(outputs, reference_outputs) 평가자를 구형 (run, example) 형태로 감싼다."""
    def wrapper(run, example):
        return {"key": fn.__name__, "score": bool(fn(run.outputs or {}, example.outputs or {}))}
    wrapper.__name__ = fn.__name__
    return wrapper


def run_evaluate(target, *, data, evaluators, experiment_prefix,
                 metadata=None, max_concurrency=1):
    kwargs = dict(
        data=data,
        experiment_prefix=experiment_prefix,
        metadata=metadata or {},
        max_concurrency=max_concurrency,   # ⚠️ VRAM 보호 — 1~2 로 제한
    )
    try:
        return evaluate(target, evaluators=evaluators, **kwargs)
    except TypeError as e:
        print(f"[!] 신형 평가자 시그니처 실패 ({e}) → 구형 (run, example) 으로 재시도 🔶")
        return evaluate(target, evaluators=[as_legacy(f) for f in evaluators], **kwargs)


def accuracy_of(results, key: str = "exact_match"):
    """실험 결과에서 평균 점수를 뽑는다. 🔶 못 뽑으면 None (웹에서 읽으면 된다)."""
    try:
        scores = []
        for row in results:
            evs = (row.get("evaluation_results") or {}).get("results") or []
            for e in evs:
                if getattr(e, "key", None) == key and getattr(e, "score", None) is not None:
                    scores.append(float(e.score))
        return sum(scores) / len(scores) if scores else None
    except Exception:
        return None


print("✅ run_evaluate / accuracy_of 준비 완료")

## 2교시 2절 ★★ — LLM-as-a-Judge: 판정자를 검증한다

규칙으로 안 되는 과제가 있습니다 — **요약 품질, 답변의 적절성, "같은 뜻인가"**.
그럴 때 채점을 모델에게 맡깁니다.

★ 그런데 핵심 질문이 남습니다.
> **"채점자가 모델이라면, 그 모델은 믿을 수 있습니까?"**

```
 예제  학생 답            로컬 4B 판정   상용 판정   사람(정답)
 ─────────────────────────────────────────────────────────────
  1    "긍정입니다"        ✅ 정답        ✅ 정답      ✅
  2    "긍정이 아닙니다"   ✅ 정답 ⚠️     ❌ 오답      ❌
  3    "약간 좋은 편"      ❌ 오답        ✅ 정답      ✅
                   │
                   ▼
⚠️ 판정자가 22% 틀린다면, 그 판정으로 잰 A/B 결과는 얼마나 믿을 수 있나?
```

> ### ★★ 이 절의 결론: 평가 결과를 믿으려면 평가자를 먼저 믿을 수 있어야 합니다
>
> **실무 절차** — 예제 10개 정도를 사람이 직접 채점해 두고 판정자와 대조합니다.
> 일치율이 낮으면 판정 프롬프트를 고치거나 판정자 모델을 바꿉니다.
> **판정자도 평가 대상입니다.**
>
> ⚠️ 판정 기준은 **이분 판정(예/아니오)** 으로 설계하십시오.
> 5점 척도는 로컬 모델에서 점수가 흔들립니다 (같은 답에 3점, 4점을 오갑니다).

In [ ]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

LOCAL_MODEL  = os.environ["MODEL"]
OPENAI_MODEL = "gpt-4o-mini"      # 🔶 예산·정책에 맞춰 확정


class Judgement(BaseModel):
    """채점 결과 — ★ bool 로 강제해야 셀 수 있다 (5주차 구조화 출력)."""

    is_correct: bool = Field(description="정답과 같은 뜻이면 true, 아니면 false")
    reason:     str  = Field(description="그렇게 판단한 이유 한 문장")


judge_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "너는 채점자다. 표현이 달라도 '같은 뜻'이면 정답으로 판정하라. "
         "부정 표현('아니다', '~가 아닙니다')이 붙어 뜻이 뒤집혔다면 오답이다. "
         "반드시 true 또는 false 하나로 판정한다."),
        ("human",
         "질문: {question}\n정답: {reference}\n학생 답: {answer}\n\n"
         "학생 답이 정답과 같은 뜻인가?"),
    ]
)

# ── 사람이 미리 채점해 둔 정답표 (Gold) ★★ ───────────────────
#    판정자를 검증하려면 '판정자보다 먼저 믿을 수 있는 기준' 이 있어야 합니다.
#    (질문, 기대 정답, 학생 답, 사람 판정)
GOLD = [
    ("배송이 하루 만에 왔어요", "긍정", "긍정", True),
    ("배송이 하루 만에 왔어요", "긍정", "긍정입니다", True),
    ("배송이 하루 만에 왔어요", "긍정", "이 리뷰는 긍정적입니다", True),
    ("배송이 하루 만에 왔어요", "긍정", "긍정이 아닙니다", False),    # ★ 부정문 함정
    ("나쁘지 않네요", "긍정", "약간 좋은 편", True),                  # ★ 규칙 기반이 못 잡는 것
    ("나쁘지 않네요", "긍정", "중립", False),
    ("화면에 흠집이 있네요", "부정", "부정적", True),
    ("화면에 흠집이 있네요", "부정", "positive", False),              # ★ 언어가 달라도 뜻으로
    ("그냥 평범합니다", "중립", "보통입니다", True),
    ("그냥 평범합니다", "중립", "긍정", False),
]


def build_judges():
    judges = [(f"로컬 {LOCAL_MODEL}",
               judge_prompt | ChatOllama(model=LOCAL_MODEL,
                                         temperature=0).with_structured_output(Judgement))]
    # 🔶 상용 판정자. 키가 없으면 조용히 건너뛴다.
    if os.getenv("OPENAI_API_KEY"):
        try:
            from langchain_openai import ChatOpenAI
            judges.append((f"상용 {OPENAI_MODEL}",
                           judge_prompt | ChatOpenAI(model=OPENAI_MODEL,
                                                     temperature=0).with_structured_output(Judgement)))
        except ImportError:
            print("[i] langchain-openai 가 없습니다")
    else:
        print("[i] OPENAI_API_KEY 가 없어 상용 판정자는 건너뜁니다.")
        print("    (로컬 판정자만으로도 '판정자도 틀린다' 는 확인됩니다) 🔶")
    return judges


def run_judge(name, judge) -> list:
    print(f"\n▶ 판정자: {name}  — 예제 {len(GOLD)}건 채점 중...")
    rows = []
    for question, reference, answer, human in GOLD:
        try:
            j = judge.invoke({"question": question, "reference": reference, "answer": answer})
            rows.append((answer, reference, bool(j.is_correct), human, j.reason))
        except Exception as e:
            print(f"  [!] 실패: {type(e).__name__} — 오답 처리")
            rows.append((answer, reference, False, human, "(판정 실패)"))
    return rows


def report(name, rows) -> float:
    agree = sum(1 for _, _, verdict, human, _ in rows if verdict == human)
    rate  = agree / len(rows)
    print(f"\n  [{name}] 사람 채점과의 일치율 : {agree}/{len(rows)} = {rate * 100:.0f}%")
    for answer, reference, verdict, human, reason in rows:
        mark = "  " if verdict == human else "⚠️"
        print(f"   {mark} 정답 {reference:<3} / 학생답 {pad(answer, 22)}"
              f" 판정 {'O' if verdict else 'X'}  사람 {'O' if human else 'X'}   {reason[:34]}")
    return rate


print("=" * 78)
print("  LLM-as-a-Judge — 판정자를 검증한다 ★★")
print("=" * 78)
print("""
왜 bool 로 강제하는가 (5주차 with_structured_output) ★
    판정 결과가 "네 맞는 것 같습니다" 같은 문장으로 오면 **집계를 할 수 없습니다.**
    is_correct: bool 로 강제해야 셀 수 있습니다.
""")

rates = {name: report(name, run_judge(name, judge)) for name, judge in build_judges()}

print("\n" + "=" * 78)
for name, rate in rates.items():
    print(f"  {name:<22} 일치율 {rate * 100:5.0f}%")
if len(rates) == 2:
    (n1, r1), (n2, r2) = rates.items()
    print(f"\n  ⚠️ 같은 답안인데 판정이 다릅니다. 차이 {abs(r1 - r2) * 100:.0f}%p")
    print(f"     {n1} 로 잰 A/B 결과와 {n2} 로 잰 A/B 결과는 **다른 숫자**가 됩니다.")
print("=" * 78)

### 무엇을 언제 쓰나

| 과제 | 권장 평가자 | 이유 |
|---|---|---|
| 분류 (긍정/부정) | 규칙 기반 | 답이 정해져 있다. 판정자를 쓸 이유가 없음 |
| 추출 (날짜·금액) | 규칙 기반 (+정규식) | 형식 검증까지 가능 |
| 형식 준수 여부 | 규칙 기반 | 5주차 파싱 실패율과 같은 발상 |
| 요약 품질 | LLM 판정자 | 규칙으로 표현 불가 |
| "같은 뜻인가" | LLM 판정자 | 의미 비교가 필요 |
| 비용·시간이 빠듯할 때 | **규칙 기반** ★ | 판정자는 예제 수만큼 호출이 추가된다 |

> 💡 **둘을 함께 쓰는 것이 실무 표준입니다.**
> 규칙 기반으로 **형식**을 보고, LLM 판정자로 **내용**을 봅니다.
> (5주차 *"스키마는 형식을 보장하지만 내용은 보장하지 않는다"* — 그 구분입니다) ★
>
> ⚠️ 오늘 실습 2·3 의 과제는 **'분류'** 입니다. 그래서 채점은 **규칙 기반(`exact_match`)** 을 씁니다.
> 판정자는 "이런 게 있고, 이런 위험이 있다" 를 보기 위해 여기서만 돌립니다. ★

## 실습 2 ★★ (2교시) — CoT vs Self-Consistency

선수과목에서 "CoT 가 낫다" 의 근거는 **논문 인용**이었습니다.
그런데 그 논문의 과제와 내 과제는 다릅니다. 모델도 다릅니다.
**내 과제에서도 CoT 가 나은지는 재봐야 압니다.**

```
같은 데이터셋 (1교시에 만든 16건)
     │
     ├──▶ 실험 A : CoT              — 단계적으로 생각한 뒤 1회 답변
     │
     └──▶ 실험 B : Self-Consistency — CoT 를 N회 샘플링 후 다수결 (5주차 실습 4)
                                         │
                                         ▼
                     비교축 3개:  정확도 · 토큰 · 지연시간   ★★
```

> ⚠️ **정확도만 보면 안 됩니다.** Self-Consistency 는 호출이 N배입니다.
> 정확도가 조금 오른 대가로 비용과 시간이 몇 배가 됩니다.
> **"그래서 쓸 것인가"** 를 판단하는 것이 오늘의 목표입니다.

### ★ 구조화 출력을 쓰면 토큰이 사라지는 문제

`with_structured_output(Result)` 의 반환값은 Pydantic 객체입니다.
편하지만 **원본 `AIMessage` 가 없어져 `usage_metadata`(토큰)를 볼 수 없습니다.**

→ `include_raw=True` 를 주면 `{"raw": AIMessage, "parsed": Result | None, ...}` 로 돌아옵니다.
원본이 함께 오므로 **토큰을 셀 수 있습니다.** ★

In [ ]:
import threading
from dataclasses import dataclass, field

INCLUDE_RAW = os.getenv("WEEK07_INCLUDE_RAW", "true").strip().lower() != "false"
# 🔶 include_raw 가 말썽이면 위 환경변수를 "false" 로 두십시오.
#    토큰은 "측정 불가" 로 처리되고 정확도·지연은 그대로 측정됩니다.


def build_chain(prompt, llm, schema):
    """prompt | llm.with_structured_output(schema) — 토큰을 보려면 include_raw=True ★"""
    if INCLUDE_RAW:
        return prompt | llm.with_structured_output(schema, include_raw=True)
    return prompt | llm.with_structured_output(schema)


def split(result):
    """invoke/batch 반환값을 (parsed, input_tokens, output_tokens) 으로 쪼갠다."""
    if isinstance(result, dict) and ("parsed" in result or "raw" in result):
        parsed = result.get("parsed")
        usage  = getattr(result.get("raw"), "usage_metadata", None) or {}
        return parsed, usage.get("input_tokens"), usage.get("output_tokens")
    return result, None, None


def label_of(parsed, default: str = "") -> str:
    """parsed 에서 label 을 안전하게 꺼낸다. 실패분은 빈 문자열 = 오답 처리."""
    value = getattr(parsed, "label", None)
    return str(value).strip() if value else default


@dataclass
class Meter:
    """실험 하나의 3축 누적기. evaluate() 가 예제를 병렬로 돌리므로 잠금이 필요하다."""

    name: str
    examples: int = 0          # 예제 수 (= traces 계산의 기준)
    llm_calls: int = 0         # 실제 LLM 호출 수 (Self-Consistency 는 예제당 N회) ★
    failures: int = 0          # 예외·파싱 실패 (오답으로 센다)
    seconds: float = 0.0
    input_tokens: int = 0
    output_tokens: int = 0
    unmeasured: int = 0        # 토큰이 안 잡힌 호출 수 🔶
    _lock: threading.Lock = field(default_factory=threading.Lock, repr=False)

    def record(self, seconds: float, usages, *, failed: bool = False) -> None:
        with self._lock:
            self.examples += 1
            self.seconds += seconds
            self.llm_calls += len(usages)
            if failed:
                self.failures += 1
            for tin, tout in usages:
                if tin is None and tout is None:
                    self.unmeasured += 1
                else:
                    self.input_tokens += tin or 0
                    self.output_tokens += tout or 0

    @property
    def total_tokens(self) -> int:
        return self.input_tokens + self.output_tokens

    @property
    def tokens_text(self) -> str:
        if self.total_tokens == 0 and self.unmeasured:
            return "측정 불가"
        suffix = f" (미측정 {self.unmeasured}건)" if self.unmeasured else ""
        return f"{self.total_tokens:,}{suffix}"

    @property
    def avg_seconds(self) -> float:
        return self.seconds / self.examples if self.examples else 0.0


def _ratio(b: float, a: float) -> str:
    return f"{b / a:.1f}배" if a else "—"


def comparison_table(meters, accuracy=None) -> str:
    """3축 비교표를 그린다. accuracy 는 {실험이름: 0.0~1.0}."""
    accuracy = accuracy or {}
    a, b = meters[0], meters[1]
    acc_a, acc_b = accuracy.get(a.name), accuracy.get(b.name)

    def pct(v):
        return f"{v * 100:.1f} %" if v is not None else "웹에서 확인"

    diff = (f"{(acc_b - acc_a) * 100:+.1f}%p"
            if acc_a is not None and acc_b is not None else "—")

    def row(axis, va, vb, delta=""):
        return "  " + pad(axis, 20) + pad(va, 18, ">") + pad(vb, 22, ">") + pad(delta, 12, ">")

    return "\n".join([
        "",
        "  3축 비교표 ★★  — RESULTS.md 에 그대로 옮겨 적으십시오",
        "  " + "─" * 72,
        row("비교축", "A. " + a.name, "B. " + b.name, "차이"),
        "  " + "─" * 72,
        row("정확도", pct(acc_a), pct(acc_b), diff),
        row("총 토큰", a.tokens_text, b.tokens_text, _ratio(b.total_tokens, a.total_tokens)),
        row("총 소요 시간", f"{a.seconds:.1f}초", f"{b.seconds:.1f}초", _ratio(b.seconds, a.seconds)),
        row("예제당 평균 지연", f"{a.avg_seconds:.1f}초", f"{b.avg_seconds:.1f}초",
            _ratio(b.avg_seconds, a.avg_seconds)),
        row("LLM 호출 수", a.llm_calls, b.llm_calls, _ratio(b.llm_calls, a.llm_calls)),
        row("traces 소모", a.llm_calls, b.llm_calls, _ratio(b.llm_calls, a.llm_calls)),
        row("실패(오답 처리)", a.failures, b.failures),
        "  " + "─" * 72,
    ])


print("✅ 3축 측정 헬퍼 준비 완료  (include_raw =", INCLUDE_RAW, ")")

### ★★ `label` 은 `Literal` 로 못 박습니다

```python
# 강의안
label: str = Field(description="긍정/부정/중립 중 하나만")
    → gemma3:4b 가 실제로  label='mixed_sentiment'  를 내놓습니다. ⚠️
      세 라벨이 아닌 값이 나오면 exact_match 가 전부 오답 처리되어
      A/B 비교 자체가 성립하지 않습니다.

# 이 노트북
label: Literal["긍정", "부정", "중립"] = Field(description="세 가지 중 하나")
    → 스키마에 enum 이 박혀 세 값 외에는 나올 수 없습니다.
```

> 📌 **`description` 은 부탁이고 `Literal` 은 계약입니다.**
> 5주차 *"'JSON으로 답해줘' vs `with_structured_output()`"* 과 **정확히 같은 구분**입니다. ★

In [ ]:
import time
from collections import Counter
from typing import Literal

MODEL = os.environ["MODEL"]

N                 = 5      # Self-Consistency 샘플 수 ⚠️ traces 가 N배로 늘어난다
TEMP_SC           = 0.8    # ★ 0 이면 5번 다 같은 답 → 다수결이 무의미
MAX_CONCURRENCY   = 1      # ⚠️ VRAM 보호. 여유가 있으면 2 🔶
BATCH_CONCURRENCY = 2      # batch 내부 동시 실행 수


class Result(BaseModel):
    """분류 결과 — reasoning 을 먼저 쓰게 해야 CoT 가 된다. ★"""

    reasoning: str = Field(description="단계별 판단 근거")
    label: Literal["긍정", "부정", "중립"] = Field(description="세 가지 중 하나")


ab_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "리뷰의 감정을 분류한다. 단계적으로 생각한 뒤 라벨을 정하라."),
        ("human",  "{review}"),
    ]
)

# ── 실험 A : CoT 1회 ──────────────────────────────────────────
chain_cot = build_chain(ab_prompt, ChatOllama(model=MODEL, temperature=0), Result)
meter_cot = Meter("cot")

# ── 실험 B : Self-Consistency (N회 다수결) ★ ──────────────────
chain_sc = build_chain(ab_prompt, ChatOllama(model=MODEL, temperature=TEMP_SC), Result)
meter_sc = Meter("self-consistency")


def target_cot(inputs: dict) -> dict:
    t0, failed = time.perf_counter(), False
    try:
        parsed, tin, tout = split(chain_cot.invoke({"review": inputs["review"]}))
        label  = label_of(parsed)
        failed = not label
        usages = [(tin, tout)]
    except Exception as e:
        # 빈 입력 등에서 깨질 수 있다. 오답으로 센다 ★
        print(f"  [!] cot 실패: {type(e).__name__}")
        label, usages, failed = "", [(None, None)], True

    elapsed = time.perf_counter() - t0
    meter_cot.record(elapsed, usages, failed=failed)
    # ★ 토큰·지연을 출력에 함께 실어 두면 웹 비교 화면에서도 보입니다.
    return {"label": label, "latency_s": round(elapsed, 2), "llm_calls": 1}


def target_sc(inputs: dict) -> dict:
    """★ 같은 입력을 N개 만들어 batch 로 넘긴다 — 이 한 줄이 Self-Consistency 의 구현이다."""
    t0 = time.perf_counter()
    rs = chain_sc.batch(
        [{"review": inputs["review"]}] * N,
        config={"max_concurrency": BATCH_CONCURRENCY},
        return_exceptions=True,        # ★ 일부가 실패해도 멈추지 않는다
    )

    labels, usages = [], []
    for r in rs:
        if isinstance(r, Exception):
            usages.append((None, None))
            continue
        parsed, tin, tout = split(r)
        usages.append((tin, tout))
        label = label_of(parsed)
        if label:
            labels.append(label)

    elapsed = time.perf_counter() - t0
    votes = Counter(labels)
    final = votes.most_common(1)[0][0] if labels else ""     # 전부 실패 → 오답 처리
    meter_sc.record(elapsed, usages, failed=not labels)

    return {
        "label": final,
        "votes": dict(votes),       # ★ 표 분포가 웹에 남는다 — 3:2 로 갈렸는지 볼 수 있다
        "latency_s": round(elapsed, 2),
        "llm_calls": N,
    }


print("✅ 실험 A/B 준비 완료 — 다음 셀에서 실행합니다 (오래 걸립니다 ⏱)")

In [ ]:
# ⏱ 여기서부터 오래 걸립니다. 실행을 걸어 두고 아래 해석 틀을 읽으십시오. ★
EXPERIMENTS = {
    "cot": (target_cot, meter_cot, {"variant": "cot", "n": 1, "temperature": 0}),
    "sc":  (target_sc,  meter_sc,  {"variant": "self-consistency", "n": N, "temperature": TEMP_SC}),
}

print("=" * 72)
print("  실습 2 — 기법 A/B 실측 (CoT vs Self-Consistency)")
print(f"  모델={MODEL}  데이터셋={DATASET_NAME}  N={N}  temp(SC)={TEMP_SC}")
print("=" * 72)

accuracy = {}
for key in ["cot", "sc"]:
    target, meter, meta = EXPERIMENTS[key]
    # ★ 설정값을 함께 기록해야 재현이 된다 (6주차 메타데이터의 연장)
    meta = dict(meta, model=MODEL, dataset=DATASET_NAME)
    print(f"\n▶ 실험 [{meter.name}] 시작 — 오래 걸립니다.")

    results = run_evaluate(
        target,
        data=DATASET_NAME,
        evaluators=[exact_match],
        experiment_prefix=meter.name,      # ★ 실험 구분 (6주차 태그의 연장)
        metadata=meta,
        max_concurrency=MAX_CONCURRENCY,
    )

    acc = accuracy_of(results)
    if acc is not None:
        accuracy[meter.name] = acc
    print(f"  [{meter.name}] 완료 — {meter.examples}건 / {meter.seconds:.1f}초 / "
          f"LLM 호출 {meter.llm_calls}회 / "
          f"정확도 {f'{acc * 100:.1f}%' if acc is not None else '웹에서 확인'}")

print(comparison_table([meter_cot, meter_sc], accuracy))

### 결과 읽기 — 웹에서 나란히 봅니다 ★★

`Datasets & Testing` → `week07-review-sentiment` → 실험 목록에서 **두 실험을 선택**
→ 예제별 비교 화면이 나옵니다. **어느 예제에서 갈렸는지** 그 자리에서 보입니다.

### 해석 연습 — 이 표를 보고 무엇을 결정합니까? ★

가정: **정확도 78% → 86% (+8%p) / 토큰 3배 / 지연 2.5배 / 비용 5배**

| 상황 | 판단 | 근거 |
|---|---|---|
| 의료·법률 판단 보조 | **채택** | 틀리면 피해가 크다. 비용은 부차적 |
| 실시간 채팅 응답 | **기각** | 지연 2.5배는 사용자가 못 기다린다 ★ |
| 하루 100만 건 배치 | **기각** | 비용 5배가 감당 불가 |
| 하루 100건 내부 도구 | **채택** | 비용 절대액이 작다 |

> 📌 **"정확도가 올랐다" 는 채택의 근거가 아닙니다.
> "무엇을 얼마에 샀는가" 를 봐야 합니다.** ★★
>
> 4주차 모델 비교표의 *"공짜는 없다"* 와 같은 구조입니다.
> 그때는 **모델 선택**, 오늘은 **기법 선택** 입니다.

> ⚠️ **두 실험의 정확도가 똑같이 나온다면** 데이터셋이 너무 쉬운 것입니다.
> `EXAMPLES` 에 **엣지 케이스를 더 넣으십시오.** 실패가 나와야 측정이 됩니다. ★

## 실습 3 (3교시) — 회귀 방지

```
[실험 A] CoT                  정확도 78%
     │
     │  "프롬프트에 '중립은 웬만하면 쓰지 마라' 를 추가하면
     │   애매한 케이스가 긍정/부정으로 잘 갈리지 않을까?"
     ▼
[실험 C] CoT + 프롬프트 수정   정확도 ??%
```

> ⚠️ **바뀐 것은 `system` 한 줄뿐입니다.** ★
> 그런데 그 한 줄이 **안 본 케이스**를 망가뜨릴 수 있습니다.

| 확인해야 할 것 | 왜 |
|---|---|
| 전체 정확도가 올랐는가 | 개선 여부 |
| **어떤 예제가 새로 틀렸는가** ★★ | **회귀 — 이게 진짜 목적** |
| 어떤 예제가 새로 맞았는가 | 개선의 실체 |

> ★ `metadata` 에 **"무엇을 바꿨는지"** 를 반드시 남기십시오.
> 실험이 10개쯤 쌓이면 `cot-v2` 라는 이름만으로는 뭘 바꿨는지 아무도 기억 못 합니다.

In [ ]:
EXPERIMENT_PREFIX = "cot-v2"
CHANGE = "중립 억제 문장 추가"       # ★ 무엇을 바꿨는지

# ── ⚠️ 바뀐 것은 system 한 줄뿐입니다 ★ ────────────────────────
prompt_v2 = ChatPromptTemplate.from_messages(
    [
        ("system",
         "리뷰의 감정을 분류한다. 단계적으로 생각한 뒤 라벨을 정하라. "
         "중립은 정말 판단이 불가능할 때만 쓴다."),        # ← 추가한 문장
        ("human", "{review}"),
    ]
)

# ★ Result 스키마는 실습 2와 **완전히 같아야** 합니다.
#   비교하려면 바뀐 것이 프롬프트 한 줄뿐이어야 합니다.
chain_v2 = build_chain(prompt_v2, ChatOllama(model=MODEL, temperature=0), Result)
meter_v2 = Meter(EXPERIMENT_PREFIX)


def target_v2(inputs: dict) -> dict:
    t0 = time.perf_counter()
    try:
        parsed, tin, tout = split(chain_v2.invoke({"review": inputs["review"]}))
        label = label_of(parsed)
        usages, failed = [(tin, tout)], not label
    except Exception as e:
        print(f"  [!] 실패: {type(e).__name__}")
        label, usages, failed = "", [(None, None)], True

    elapsed = time.perf_counter() - t0
    meter_v2.record(elapsed, usages, failed=failed)
    return {"label": label, "latency_s": round(elapsed, 2), "llm_calls": 1}


print("=" * 72)
print("  실습 3 — 회귀 방지 재평가")
print(f"  바꾼 것: {CHANGE}   (system 한 줄)")
print("=" * 72)

results_v2 = run_evaluate(
    target_v2,
    data=DATASET_NAME,                 # ★ 같은 데이터셋 — 이게 핵심
    evaluators=[exact_match],
    experiment_prefix=EXPERIMENT_PREFIX,
    metadata={                          # ★ 무엇을 바꿨는지 기록
        "variant": "cot", "version": "v2", "change": CHANGE,
        "model": MODEL, "temperature": 0,
    },
    max_concurrency=MAX_CONCURRENCY,
)

acc_v2 = accuracy_of(results_v2)
print(f"\n[{EXPERIMENT_PREFIX}] 완료 — {meter_v2.examples}건 / {meter_v2.seconds:.1f}초 / "
      f"토큰 {meter_v2.tokens_text} / 정확도 "
      f"{f'{acc_v2 * 100:.1f}%' if acc_v2 is not None else '웹에서 확인'}")

In [ ]:
# ★ v1(cot) 과 v2(cot-v2) 를 예제별로 대조해 **회귀**를 찾는다.
#   🔶 실험 결과를 코드로 읽는 API 는 버전차가 있습니다.
#      실패하면 웹에서 보십시오 (Datasets & Testing → 두 실험 선택 → 예제별 비교).

projects = list(client.list_projects(reference_dataset_name=DATASET_NAME))


def latest(match):
    found = [p for p in projects if match(p.name)]
    return max(found, key=lambda p: p.start_time) if found else None


# ⚠️ "cot-" 은 "cot-v2-..." 도 잡습니다. v2 를 명시적으로 제외합니다.
v1 = latest(lambda n: n.startswith("cot-") and not n.startswith(EXPERIMENT_PREFIX))
v2 = latest(lambda n: n.startswith(EXPERIMENT_PREFIX))

if not v1 or not v2:
    print("[!] 비교할 실험을 찾지 못했습니다. 실습 2를 먼저 돌리십시오.")
    print("    (또는 웹 비교 화면에서 직접 보십시오 — 그게 원래 방식입니다) ★")
else:
    def review_of(payload: dict) -> str:
        """루트 Run 의 입력에서 review 를 꺼낸다. 🔶 감싸는 형태가 버전마다 다르다."""
        payload = payload or {}
        if "review" in payload:
            return str(payload["review"])
        inner = payload.get("inputs")
        return str(inner.get("review", "")) if isinstance(inner, dict) else str(payload)

    def collect(project):
        return {review_of(r.inputs): (r.outputs or {}).get("label", "")
                for r in client.list_runs(project_name=project.name, is_root=True)}

    gold = {(e.inputs or {}).get("review", ""): (e.outputs or {}).get("label", "")
            for e in client.list_examples(dataset_name=DATASET_NAME)}
    a, b = collect(v1), collect(v2)

    print("=" * 78)
    print(f"  예제별 대조 :  {v1.name}  vs  {v2.name}")
    print("=" * 78)
    print("  " + pad("예제", 28) + pad("정답", 6) + pad("v1", 8) + pad("v2", 8) + "판정")
    print("-" * 78)

    tally = {"유지": 0, "개선": 0, "회귀": 0, "둘다오답": 0}
    for review, want in gold.items():
        if review not in a or review not in b:
            continue
        ok1, ok2 = a[review].strip() == want, b[review].strip() == want
        kind = "유지" if ok1 and ok2 else "개선" if ok2 else "회귀" if ok1 else "둘다오답"
        verdict = {"유지": "유지", "개선": "개선 ★",
                   "회귀": "회귀 ⚠️★★", "둘다오답": "둘다오답"}[kind]
        tally[kind] += 1
        shown = (review or "(빈 입력)")[:26]
        print("  " + pad(shown, 28) + pad(want, 6) + pad(a[review][:6], 8)
              + pad(b[review][:6], 8) + verdict)

    print("-" * 78)
    print(f"  유지 {tally['유지']} / 개선 {tally['개선']} / "
          f"⚠️ 회귀 {tally['회귀']} / 둘다오답 {tally['둘다오답']}")
    print("=" * 78)

### 비교 화면에서 회귀 찾기 ★★

```
예제        v1      v2      판정
────────────────────────────────────────
#1 배송..    ✅      ✅      유지
#3 나쁘지..  ❌      ✅      개선 ★
#7 그냥..    ✅      ❌      회귀 ⚠️★★   ← 이걸 찾는 게 목적
#9 가격은..  ❌      ✅      개선
────────────────────────────────────────
전체        78%  →  84%
```

> ⚠️⚠️ **전체 정확도만 보면 #7 을 놓칩니다.** 78% → 84% 면 "성공"으로 보입니다.
> 그런데 **원래 잘 되던 케이스 하나가 망가졌습니다.**
> 그 케이스가 서비스에서 가장 흔한 입력이라면?
> **전체 수치는 올랐는데 사용자 만족은 떨어집니다.**

| 결과 | 조치 |
|---|---|
| 개선만 있고 회귀 없음 | 채택 |
| 개선 > 회귀, 회귀가 사소함 | 채택 + 회귀 케이스를 데이터셋에 명시 |
| 개선 < 회귀 | 되돌린다 |
| **회귀 케이스가 중요한 입력** ★ | **수치가 올라도 재검토** |

> 📌 이것이 **회귀 테스트**입니다. 코드를 고칠 때마다 테스트를 돌리는 것과 정확히 같습니다.
> 차이는 테스트가 통과/실패가 아니라 **'점수'** 라는 것뿐입니다.
>
> 💡 **실무 루틴**
> 프롬프트 수정 → 같은 데이터셋 재평가 → 회귀 확인 → 새 실패는 데이터셋에 추가 → 반복
> **데이터셋은 쓸수록 강해집니다.**

## 3교시 2절 — 무료 한도 계산

```
소모량 ≒ 예제 수 × 실험 수 × (샘플링 배수) + 판정자 호출
```

> ⚠️ **주의해야 할 조합**
> Self-Consistency(N=5) × 예제 100개 × 실험 5회 = **2,500 traces**
> 한 번에 한도의 절반을 씁니다. **N 과 예제 수는 곱해집니다.** ★

In [ ]:
FREE_MONTHLY = 5_000        # 무료 Developer 플랜: 월 5,000 traces

# (이름, 예제 수, 샘플링 배수, 설명)
TODAY = [
    ("실습 2  A (CoT)", 16, 1, "예제당 1회"),
    ("실습 2  B (Self-Consistency)", 16, 5, "예제당 5회 ⚠️"),
    ("실습 3  C (CoT-v2)", 16, 1, "같은 데이터셋 재평가"),
    ("2교시 판정자 시연", 10, 1, "로컬 판정자"),
]
LATER = [
    ("보충 Zero-shot", 16, 1, "수업 후 · 무배점"),
    ("보충 Few-shot", 16, 1, "수업 후 · 무배점"),
    ("보충 Step-Back", 16, 2, "상위 질문 1회 + 원 질문 1회 ★"),
]


def table(title: str, rows) -> int:
    def line(name, n, mult, traces, note=""):
        return ("  " + pad(name, 34) + pad(n, 6, ">") + pad(mult, 6, ">")
                + pad(traces, 10, ">") + ("   " + note if note else ""))

    print(f"\n  {title}")
    print("  " + "─" * 62)
    print(line("항목", "예제", "배수", "traces"))
    print("  " + "─" * 62)
    total = 0
    for name, n, mult, note in rows:
        total += n * mult
        print(line(name, n, mult, n * mult, note))
    print("  " + "─" * 62)
    print(line("소계", "", "", total))
    return total


print("=" * 66)
print("  무료 한도 소모량 계산 — 학생 1인 기준")
print("=" * 66)
today = table("오늘 수업", TODAY)
later = table("수업 후 보충 실습 (무배점)", LATER)

print("\n  " + "=" * 62)
for name, value in [("무료 한도 (월)", FREE_MONTHLY), ("오늘 수업", today),
                    ("보충 실습", later), ("합계", today + later)]:
    print("  " + pad(name, 40) + pad(f"{value:,}", 10, ">"))
print("  " + pad("남는 여유", 40) + pad(f"{FREE_MONTHLY - today - later:,}", 10, ">") + "   ✅ 충분")
print("  " + "=" * 62)

In [ ]:
# ── 직접 계산해 보십시오 — 값을 바꿔 가며 실행 ──
N_EXAMPLES, N_RUNS, N_MULT = 100, 5, 5      # 예제 100 × 실험 5회 × 샘플링 5배 ⚠️

traces = N_EXAMPLES * N_RUNS * N_MULT
print("=" * 66)
print(f"  예제 {N_EXAMPLES}개 × 실험 {N_RUNS}회 × 샘플링 {N_MULT}배 = {traces:,} traces")
print(f"  무료 한도 {FREE_MONTHLY:,} 대비 {traces / FREE_MONTHLY * 100:.1f}%")
print("=" * 66)
if traces > FREE_MONTHLY:
    print("  ⚠️ 한도 초과입니다. 429 로 거부되며 기록이 남지 않습니다.")
elif traces > FREE_MONTHLY * 0.2:
    print("  ⚠️ 한 번에 한도의 20% 이상을 씁니다. 예제 수나 N 을 줄이십시오. ★")
else:
    print("  ✅ 여유 있습니다.")

print("""
한도 관리 3원칙
  ① 데이터셋을 작게       10~20개. 늘리는 건 언제든 가능
  ② 샘플링 배수를 조심    batch(N) 은 traces 가 N배 ★
  ③ 디버깅 중에는 끄기    LANGSMITH_TRACING=false

⚠️ 추적 보존은 14일입니다 (6주차). 오늘 만든 실험 결과도 2주 뒤 사라집니다.
   **캡처를 지금 저장**하십시오. 중간고사 대비 자료로 쓸 것입니다. ★
""")

## 보충 (수업 후 · 무배점) — Zero-shot / Few-shot / Step-Back

수업 중에는 실험군 2개(CoT / Self-Consistency)만 돌립니다.
25분 안에 4~5개를 돌리면 **어느 것도 제대로 끝나지 않습니다.**

| 실험군 | 프롬프트 | 예상되는 특징 |
|---|---|---|
| Zero-shot | 지시만 | 가장 싸고 빠름. **기준선(baseline)** |
| Few-shot | 예시 3~5개 첨부 | 형식 준수↑, 입력 토큰↑ |
| **Step-Back** ★ | 한 발 물러선 질문을 먼저 | **선수과목 미학습 기법** |

### ★ Step-Back Prompting

```
원 질문: "2023년 A사 B제품 출시일에 경쟁사는 무엇을 했나?"
     │
     │  ① 한 발 물러선 질문을 먼저 만든다
     │     → "A사 B제품은 언제 출시되었나?"
     │  ② 그 답(일반적·상위 사실)을 근거로
     │  ③ 원 질문에 답한다
     ▼
구체적 질문에 바로 답하면 틀리기 쉬운 문제를, 상위 사실부터 확보해 푼다
```

> ★ 구조적으로는 5주차 **Least-to-Most 의 사촌**입니다 — 둘 다 **직렬 분해**.
> 차이는 *"쉬운 것부터 순서대로"*(Least-to-Most) vs *"한 단계 추상화된 질문 먼저"*(Step-Back).
>
> ⚠️ Step-Back 은 예제당 LLM 호출이 **2회**입니다. traces 도 2배입니다.
> ⚠️ 중간고사에 출제하지 않습니다 (수행 여부가 학생마다 다르므로).

In [ ]:
from langchain_core.output_parsers import StrOutputParser

llm_zero = ChatOllama(model=MODEL, temperature=0)


class Label(BaseModel):
    """Zero-shot·Few-shot 은 근거를 요구하지 않는다 — 그래서 출력 토큰이 적다. ★"""

    label: Literal["긍정", "부정", "중립"] = Field(description="세 가지 중 하나")


# ── ① Zero-shot — 지시만. 기준선(baseline) ────────────────────
zeroshot_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "리뷰의 감정을 긍정/부정/중립 중 하나로 분류한다."),
        ("human",  "{review}"),
    ]
)

# ── ② Few-shot — 예시를 붙인다 (5주차) ────────────────────────
#     ⚠️ 데이터셋에 들어 있는 예제를 예시로 쓰면 안 됩니다
#        — 답을 알려주고 채점하는 꼴입니다 ★★
FEWSHOT_EXAMPLES = [
    ("포장이 꼼꼼해서 좋았습니다", "긍정"),
    ("설명과 다른 물건이 왔어요", "부정"),
    ("어제 수령했습니다", "중립"),
    ("싸지는 않지만 만족합니다", "긍정"),
]
fewshot_prompt = ChatPromptTemplate.from_messages(
    [("system", "리뷰의 감정을 긍정/부정/중립 중 하나로 분류한다.")]
    + [m for review, label in FEWSHOT_EXAMPLES for m in (("human", review), ("ai", label))]
    + [("human", "{review}")]
)

# ── ③ Step-Back — 상위 질문을 먼저 ★ ─────────────────────────
stepback_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "너는 리뷰 분석가다. 감정 라벨은 아직 정하지 마라. "
         "이 리뷰가 무엇에 대해(배송·품질·가격·서비스 등) 어떤 태도를 보이는지 "
         "한 발 물러서서 두 문장으로만 정리하라."),
        ("human", "{review}"),
    ]
)
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "아래 분석을 근거로 리뷰의 감정을 긍정/부정/중립 중 하나로 분류한다. "
         "장점과 단점이 함께 있으면 마지막 절의 인상을 따른다."),
        ("human", "리뷰: {review}\n\n분석: {analysis}\n\n라벨은?"),
    ]
)

# 한 건만 눈으로 확인해 봅니다 (전체 평가는 각자 수행하십시오)
SAMPLE = "가격은 좋은데 품질은 실망입니다"

print("── Zero-shot ──")
print((zeroshot_prompt | llm_zero.with_structured_output(Label)).invoke({"review": SAMPLE}))

print("\n── Few-shot ──")
print((fewshot_prompt | llm_zero.with_structured_output(Label)).invoke({"review": SAMPLE}))

print("\n── Step-Back (2회 호출) ──")
analysis = (stepback_prompt | llm_zero | StrOutputParser()).invoke({"review": SAMPLE})
print("  [1단계 상위 분석]", analysis.replace("\n", " ")[:120])
print("  [2단계 최종 라벨]",
      (final_prompt | llm_zero.with_structured_output(Label))
      .invoke({"review": SAMPLE, "analysis": analysis}))

## 오늘 확인할 것

- [ ] 🔶 `evaluate()` 시그니처를 사전 점검 셀로 확정했다 ★★
- [ ] 데이터셋 16건을 등록하고 **내 엣지 케이스 2~3개**를 추가했다
- [ ] `contains` 가 `"긍정이 아닙니다"` 를 정답으로 세는 것을 봤다 ★★
- [ ] 판정자의 **사람 채점 일치율**을 재 봤다 ★★
- [ ] **CoT vs Self-Consistency 를 3축으로** 비교했다 ★★
- [ ] 프롬프트 한 줄 수정 후 **회귀**를 찾았다 ★★
- [ ] 무료 한도 소모량을 계산했다

### 📌 과제 (신규 과제 없음 — 다음 과제는 9주차)

| # | 할 것 |
|---|---|
| 1 | **3축 비교표**를 `RESULTS.md` 로 정리해 `week07/` 에 커밋 |
| 2 | **실험 비교 화면 캡처 2장** (A/B 비교 · 회귀 대조) ★ |
| 3 | 보충 실습 (Zero-shot / Few-shot / Step-Back) — 무배점 |
| 4 | ★ **실습 코드를 직접 다시 돌려 볼 것** — 중간고사에 **코드 읽기 문항**이 있습니다 |

> ⚠️ **추적 보존은 14일입니다.** 중간고사가 10/23 이므로 **캡처를 반드시 남기십시오.** ★

### 오늘의 한 줄

> **"좋아진 것 같아요" 는 근거가 아니다.**
>
> 그리고 — **"정확도가 올랐다" 도 채택의 근거가 아니다.
> "무엇을 얼마에 샀는가" 를 봐야 한다.** ★★